# VibeThinker-3B — Hands-On Notebook

[`WeiboAI/VibeThinker-3B`](https://huggingface.co/WeiboAI/VibeThinker-3B) · [GitHub](https://github.com/WeiboAI/VibeThinker) · [Technical Report (arXiv:2606.16140)](https://arxiv.org/abs/2606.16140)

**VibeThinker-3B** is a 3B-parameter reasoning model (fine-tuned from `Qwen2.5-3B`) built for *verifiable* reasoning — competition math (AIME, HMMT, IMO-AnswerBench), coding (LiveCodeBench, LeetCode), and STEM. Despite its small size it reaches the accuracy range of much larger frontier models on these specific, verifiable domains.

The core idea behind the model is the **Parametric Compression-Coverage Hypothesis**: multi-step reasoning with reliable feedback signals is a *compressible*, parameter-dense capability, so a compact model can carry near-frontier reasoning ability — while broad open-domain world knowledge still benefits from scale.

### What this notebook covers
1. Environment setup & requirements
2. Loading the model and tokenizer
3. A reusable `VibeThinker` inference wrapper (matching the official recipe)
4. Running math, coding, and STEM examples
5. Streaming generation
6. Inspecting the reasoning trace
7. Batched inference
8. (Optional) high-throughput serving with vLLM
9. Practical tips & gotchas

> **Hardware note:** This is a 3B model in BF16 (~6 GB weights). A GPU with ≥12 GB VRAM is comfortable. It runs on CPU but reasoning generations are *long* (up to 100K+ tokens) and will be slow.

## 1. Environment Setup

The model requires **`transformers >= 4.54.0`**. For high-throughput inference the authors recommend **vLLM 0.10.1** or **SGLang >= 0.4.9.post6** (covered in the optional section near the end).

Run the cell below once. If you're on Colab/Kaggle, restart the runtime afterward if `transformers` was upgraded.

In [ ]:
# Install / upgrade required packages
%pip install -q "transformers>=4.54.0" "accelerate>=0.30.0" torch --upgrade

# Optional: faster downloads from the Hugging Face Hub
%pip install -q "huggingface_hub[hf_transfer]"

In [1]:
import os
# Enable the faster Rust-based downloader (optional)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

import torch
import transformers

print("PyTorch:        ", torch.__version__)
print("Transformers:   ", transformers.__version__)
print("CUDA available: ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:            ", torch.cuda.get_device_name(0))
    print("VRAM (GB):      ", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:294: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


PyTorch:         2.11.0+cu128
Transformers:    5.10.2
CUDA available:  True
GPU:             Tesla T4
VRAM (GB):       15.6


## 2. Load the Model & Tokenizer

VibeThinker-3B is a standard causal LM, so we load it with `AutoModelForCausalLM`. We use:

- `torch_dtype="bfloat16"` — the model's native precision
- `device_map="auto"` — place weights on GPU automatically
- `low_cpu_mem_usage=True` — reduce peak host RAM during load

The first run downloads \~6 GB of weights and caches them under `~/.cache/huggingface`.

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_PATH = "WeiboAI/VibeThinker-3B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    low_cpu_mem_usage=True,
    torch_dtype="bfloat16",
    device_map="auto",
)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded {MODEL_PATH}")
print(f"Parameters: {n_params/1e9:.2f}B")
print(f"Device:     {model.device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.64k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

Loaded WeiboAI/VibeThinker-3B
Parameters: 3.09B
Device:     cuda:0


## 3. A Reusable Inference Wrapper

The model expects prompts formatted with its **chat template** (the same Qwen2.5-style template it was trained on). The class below mirrors the official Quick Start code and exposes a clean `infer_text()` method.

### Recommended sampling parameters
The technical report evaluates with these settings — **use them for best results**:

| Param | Value | Note |
|-------|-------|------|
| `temperature` | `1.0` | Higher than usual; the model is trained for diverse-then-verified reasoning |
| `top_p` | `0.95` | |
| `top_k` | `-1` / `None` | Disabled |
| `max_new_tokens` | up to `~100K` | Reasoning traces can be very long |

> Lowering `temperature` toward 0 tends to *hurt* this model on reasoning tasks — it was tuned for `temperature=1.0`.

In [3]:
from transformers import GenerationConfig

class VibeThinker:
    """Thin wrapper around VibeThinker-3B matching the official inference recipe."""

    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer

    def build_prompt(self, prompt, system=None):
        messages = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})
        return self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    def infer_text(self, prompt, system=None, max_new_tokens=8192,
                   temperature=1.0, top_p=0.95, top_k=None):
        text = self.build_prompt(prompt, system=system)
        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

        generation_config = GenerationConfig(
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
        )
        generated_ids = self.model.generate(
            **model_inputs,
            generation_config=generation_config,
        )
        # Strip the prompt tokens, keep only the new generation
        new_ids = [
            out[len(inp):]
            for inp, out in zip(model_inputs.input_ids, generated_ids)
        ]
        return self.tokenizer.batch_decode(new_ids, skip_special_tokens=True)[0]

vibe = VibeThinker(model, tokenizer)
print("Wrapper ready.")

Wrapper ready.


### Peek at the formatted prompt
It helps to see exactly what gets fed to the model after applying the chat template.

In [4]:
print(vibe.build_prompt("What is 2 + 2?"))

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
What is 2 + 2?<|im_end|>
<|im_start|>assistant



## 4. Example A — Competition Math

VibeThinker-3B's home turf. We give it an AIME-style problem. Note that we keep `max_new_tokens` moderate here so the cell finishes quickly; for genuinely hard problems you may want to raise it substantially.

In [5]:
math_problem = (
    "Find the number of ordered pairs of positive integers (a, b) such that "
    "a + b = 1000 and neither a nor b has a zero digit. "
    "Show your reasoning and give the final answer."
)

answer = vibe.infer_text(math_problem, max_new_tokens=4096)
print(answer)

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


<think>We are asked: number of ordered pairs (a,b) of positive integers such that a+b=1000 and neither a nor b has a zero digit. The digits of a and b (in decimal representation) cannot contain the digit zero.

We need to count ordered pairs (a,b) where a>=1, b>=1, a+b=1000, both a and b have no zero digit.

We need to consider positive integers a and b between 1 and 999 inclusive (since both positive and sum=1000). a from 1 to 999, b=1000-a.

We need to count a in [1,999] such that both a and 1000-a have no digit zero.

Ordered pairs means (a,b) distinct from (b,a) unless a=b. But since a+b=1000, if a ≠ 500, then (a,b) and (b,a) both correspond to different a values, as each a uniquely determines the ordered pair (a,1000-a). So counting a that satisfy condition yields the number of ordered pairs (a,b). So count number of a in 1..999 where both a and 1000-a contain no zero digits.

We can approach via direct enumeration or combinatorial counting using digit constraints.

Observation: 1

## 5. Example B — Coding

The model passes a high fraction of unseen LeetCode contest problems on the first attempt. Let's ask for a concrete implementation.

In [8]:
coding_problem = (
    "Write a Python function `longest_palindromic_substring(s: str) -> str` "
    "that returns the longest palindromic substring of `s` in O(n) time using "
    "Manacher's algorithm. Include a short explanation and a couple of test cases."
)

solution = vibe.infer_text(coding_problem, max_new_tokens=2048)
print(solution)

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


<think>The user wants a Python function `longest_palindromic_substring(s: str) -> str` that returns the longest palindromic substring of s in O(n) time using Manacher's algorithm. Also include a short explanation and a couple of test cases.

We need to provide code implementing Manacher's algorithm. Should be in Python, with function signature as requested.

We should also explain the algorithm briefly and give some test cases.

We need to avoid any disallowed content. The request is straightforward.

Thus output code, explanation, test cases. Provide everything in a single answer. The code must be correct: we need to implement Manacher's algorithm for both even and odd length palindromes in a unified way.

One typical approach: transform the string by inserting separators (like '#') between characters and at ends (e.g., '^#a#b#c#$' etc.) Then compute the array P where P[i] is the radius of the palindrome centered at i (the number of characters to each side including separators). Then 

## 6. Example C — STEM Reasoning

A physics/chemistry-flavored question with a verifiable numeric answer.

In [7]:
stem_problem = (
    "A 0.50 kg block slides down a frictionless incline of height 1.2 m, "
    "starting from rest. Using energy conservation, what is its speed at the "
    "bottom? Take g = 9.8 m/s^2. Give the steps and the final numeric answer."
)

print(vibe.infer_text(stem_problem, max_new_tokens=2048))

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


<think>The problem: a 0.50 kg block slides down a frictionless incline of height 1.2 m, starting from rest. Using energy conservation, find its speed at the bottom. g = 9.8 m/s^2.

We must provide steps and final numeric answer in \boxed{}.

We need to reason: initial potential energy mgh at top, final kinetic energy (1/2) m v^2 at bottom (assuming zero friction). So m g h = (1/2) m v^2 -> v = sqrt(2 g h). The mass cancels. So v = sqrt(2 * 9.8 * 1.2) = sqrt(2 * 9.8 * 1.2). Compute: 2 * 9.8 = 19.6; *1.2 = 19.6*1.2 = 23.52. sqrt(23.52) = approximate. sqrt(23.52) ~ 4.849... Let's compute: 4.85 m/s approx. Let's do more precise: sqrt(23.52). 4.85^2 = 23.5225, so indeed about 4.85 m/s.

Thus answer: about 4.85 m/s. Put in \boxed{4.85\text{ m/s}} perhaps.

We need step-by-step reasoning. Also mass 0.50 kg is irrelevant but we can mention.

Thus final answer in box.

Ok.</think>**Step‑by‑step solution**

1. **Write the energy conservation equation.**  
   Since the incline is frictionless, me

## 7. Streaming Generation

For interactive use you'll often want tokens to appear as they're produced. `TextIteratorStreamer` runs `generate()` on a background thread and yields text chunks.

In [9]:
from transformers import TextIteratorStreamer
from threading import Thread

def stream_infer(prompt, system=None, max_new_tokens=2048,
                 temperature=1.0, top_p=0.95):
    text = vibe.build_prompt(prompt, system=system)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    streamer = TextIteratorStreamer(
        tokenizer, skip_prompt=True, skip_special_tokens=True
    )
    gen_kwargs = dict(
        **inputs,
        streamer=streamer,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        top_k=None,
    )
    thread = Thread(target=model.generate, kwargs=gen_kwargs)
    thread.start()

    collected = []
    for chunk in streamer:
        print(chunk, end="", flush=True)
        collected.append(chunk)
    thread.join()
    return "".join(collected)

_ = stream_infer("Explain why the square root of 2 is irrational.", max_new_tokens=1024)

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


<think>The user asks: "Explain why the square root of 2 is irrational. Please reason step by step, and put your final answer within \boxed{}."

We need to produce a clear explanation of why sqrt(2) is irrational, presumably using classic proof by contradiction, maybe referencing density of rational numbers, prime factorization, etc.

We must obey content policy. It's a standard math proof. There's no problem. We need to give a step-byproof.

The final answer (the conclusion) must be in a \boxed{}.

Probably they want something like "sqrt(2) is irrational". So final answer: \boxed{\text{irrational}} or \boxed{\text{sqrt(2) is irrational}}.

Better: The final answer is something like \boxed{\sqrt{2}\text{ is irrational}}.

We need to respond with reasoning, then final answer in \boxed{}.

Make sure to follow guidelines.

We need to respond in a helpful way. The answer can include LaTeX. The final answer in \boxed{}.

We can also discuss the classic Euclid's proof: Assume sqrt(2) = a/b in

## 8. Inspecting the Reasoning Trace

Reasoning models often separate their *thinking* from the *final answer*. Depending on the version, VibeThinker may wrap intermediate reasoning in `<think>...</think>` tags. The helper below splits a response into the reasoning portion and the final answer if such tags are present.

In [10]:
import re

def split_thinking(text):
    """Separate <think>...</think> reasoning from the final answer, if present."""
    match = re.search(r"<think>(.*?)</think>(.*)", text, flags=re.DOTALL)
    if match:
        thinking = match.group(1).strip()
        final = match.group(2).strip()
        return thinking, final
    return None, text.strip()

sample = vibe.infer_text(
    "If 3 cats catch 3 mice in 3 minutes, how many cats are needed to "
    "catch 100 mice in 100 minutes?",
    max_new_tokens=2048,
)

thinking, final = split_thinking(sample)
if thinking:
    print("===== REASONING (truncated) =====")
    print(thinking[:1500], "...\n")
print("===== FINAL ANSWER =====")
print(final)

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


===== REASONING (truncated) =====
We have a problem: "If 3 cats catch 3 mice in 3 minutes, how many cats are needed to catch 100 mice in 100 minutes?" This seems like a simple proportion: In 3 minutes, 3 cats catch 3 mice. So each cat catches one mouse in 3 minutes. So the catch rate per cat = 1 mouse per 3 minutes = 0.333... mice per minute. So to catch 100 mice, we need total catch rate * time * number of cats = number of mice. Let C = number of cats, t=100 minutes, rate per cat r = 1/3 mouse per minute. Then total caught = C * r * t = C * (1/3) * 100 = (100C)/3. Set equal to 100 => 100C/3 = 100 => C = 3. So answer is 3 cats. However, maybe there is nuance: perhaps each cat catches one mouse per 3 minutes (like the first scenario). So after 3 minutes they each catch one. In 100 minutes, each cat can catch (100/3) mice, but mice are discrete. Possibly need to round up. But the question likely expects linear scaling: 3 cats for 3 mice in 3 minutes. That yields a rate of 1 mouse per cat

## 9. Batched Inference

Process several prompts in one `generate()` call. Make sure padding is configured correctly (left padding is required for correct decoder generation).

In [11]:
# Configure left padding for batched generation
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

prompts = [
    "What is the 10th Fibonacci number?",
    "Solve for x: 2x + 7 = 19.",
    "What is the derivative of x^3 with respect to x?",
]

texts = [vibe.build_prompt(p) for p in prompts]
batch = tokenizer(texts, return_tensors="pt", padding=True).to(model.device)

with torch.no_grad():
    out = model.generate(
        **batch,
        max_new_tokens=1024,
        do_sample=True,
        temperature=1.0,
        top_p=0.95,
        top_k=None,
    )

for i, (p, seq) in enumerate(zip(prompts, out)):
    gen = seq[batch.input_ids.shape[1]:]
    decoded = tokenizer.decode(gen, skip_special_tokens=True)
    print(f"\n----- Prompt {i+1}: {p} -----")
    print(decoded[-600:])  # show the tail (final answer)

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



----- Prompt 1: What is the 10th Fibonacci number? -----
t common answer: 55.

Thus final answer: \boxed{55}.

Ok, answer.

But must be careful: Provide step-by-step reasoning and final answer in \boxed{}.

Let's do:

1. Define the Fibonacci sequence: F1=1, F2=1, Fn = F_{n-1} + F_{n-2} for n > 2.

2. Compute: F3 = 2, F4 = 3, F5 = 5, F6 = 8, F7 = 13, F8 = 21, F9 = 34, F10 = 55.

Thus answer: \boxed{55}.

Alternatively, if they define F0=0, then F10 = 55 as well.

Thus answer: \boxed{55}.</think>The Fibonacci numbers are defined by  

\[
F_1 = 1,\qquad F_2 = 1,\qquad 
F_n = F_{n-1}+F_{n-2}\;(n\ge 3).
\]

Now compute successively:

\[
\begin{aligned}
F

----- Prompt 2: Solve for x: 2x + 7 = 19. -----
ep, and put your final answer within \boxed{}." So we need to give reasoning: step by step.

Thus final answer \boxed{6}. Ensure to follow format.

Thus we need to produce final answer. Possibly they'd want \boxed{6}.

So I will answer accordingly.</think>To solve the equation \(2x + 7 = 19\

## 10. Practical Tips & Guidelines

- **Use the recommended sampling settings** (`temperature=1.0`, `top_p=0.95`, `top_k` disabled). This model is *not* tuned for greedy/low-temperature decoding, and lowering temperature can degrade reasoning accuracy.
- **Give it room to think.** Reasoning traces can run to tens of thousands of tokens. If answers look cut off, raise `max_new_tokens`. The full context window is 64K.
- **Play to its strengths.** It excels at *verifiable* tasks — math, coding, STEM — where answers can be checked. For broad open-domain knowledge or general chit-chat, a larger general-purpose model is usually a better fit.
- **Sampling is stochastic.** With `temperature=1.0`, repeated runs differ. For competition-style eval, the report uses test-time scaling (e.g. sampling many candidates + Claim-Level Reliability Assessment) rather than a single sample.
- **Memory.** BF16 weights are ~6 GB; with activations and KV cache for long generations, budget ≥12 GB VRAM. Use vLLM for efficient long-context KV-cache management.
- **License.** The model is released under the **MIT License**.

### Citation
```bibtex
@misc{xu2026vibethinker3b,
  title={VibeThinker-3B: Exploring the Frontier of Verifiable Reasoning in Small Language Models},
  author={Sen Xu and Shixi Liu and Wei Wang and Jixin Min and Yingwei Dai and
          Zhibin Yin and Yirong Chen and Xin Zhou and Junlin Zhang},
  year={2026},
  eprint={2606.16140},
  archivePrefix={arXiv},
  primaryClass={cs.AI},
  url={https://arxiv.org/abs/2606.16140},
}
```